In [ ]:
import re
import json
import tempfile
import os
import shutil
import subprocess
from pathlib import Path
from dataclasses import dataclass
from itertools import islice
from tqdm import tqdm
from joblib import Parallel, delayed


@dataclass
class AssemblyBlock:
    symbol_name: str
    filepath: str | None
    raw_instructions: list[str]

    def original_format(self) -> str:
        return f"TEXT {self.symbol_name} {self.filepath}\n" + "\n".join(self.raw_instructions)

    def labeled_format(self) -> str:
        return self.symbol_name + "\n" + clean_instructions(self.raw_instructions)


def extract_go_packages(code: str) -> list[str]:
    pattern = r'import\s*\(([\s\S]*?)\)|import\s*["\\'](.*?)["\\'"]'
    matches = re.findall(pattern, code)
    packages = []
    for multi_import, single_import in matches:
        if multi_import:
            for line in multi_import.splitlines():
                line = line.strip().strip('"').strip("'")
                if line:
                    if ' "' in line:
                        line = line.split(' "')[1]
                    if '" ' in line:
                        line = line.split('" ')[0]
                    packages.append(line)
        elif single_import:
            packages.append(single_import.strip())
    return packages


def is_internal_package(pkg: str) -> bool:
    return pkg.split('/')[0].count('.') == 0


def install_go_packages(code: str, tmp: str) -> bool:
    for pkg in extract_go_packages(code):
        if not is_internal_package(pkg):
            if subprocess.run(
                ["go", "get", pkg],
                cwd=tmp,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.PIPE
            ).returncode != 0:
                return False
    return True


def get_compiled_code(code: str, category: str, breadcrumbs: str,
                      tmp_root: str = 'tmp/', gcflags: str | None = None) -> bytes | None:
    try:
        os.makedirs(tmp_root, exist_ok=True)
        with tempfile.TemporaryDirectory(dir=tmp_root, prefix=breadcrumbs, delete=False) as tmp:
            code_file = "test.go"
            code_path = os.path.join(tmp, code_file)
            with open(code_path, "w", encoding="utf-8") as f:
                f.write(code)
            output_path = code_path + ".out"

            if subprocess.run(
                ["go", "mod", "init", "example.com/test"],
                cwd=tmp, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, encoding="utf-8"
            ).returncode != 0:
                return None

            if subprocess.run(
                ["goimports", "-w", code_file],
                cwd=tmp, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, encoding="utf-8"
            ).returncode != 0:
                return None

            if not install_go_packages(code, tmp):
                return None

            cmd = ["go", "build"]
            if gcflags:
                cmd += ["-gcflags", gcflags]
            cmd += ["-o", output_path]
            if category == "plugin":
                cmd += ["-buildmode=plugin"]
            cmd += [code_file]

            if subprocess.run(
                cmd, cwd=tmp, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, encoding="utf-8"
            ).returncode != 0:
                return None

            with open(output_path, "rb") as f:
                binary_data = f.read()
    except Exception:
        return None

    try:
        shutil.rmtree(tmp)
    except Exception:
        pass

    return binary_data


def get_assembly_code(binary_bytes: bytes) -> str | None:
    with tempfile.NamedTemporaryFile(delete=False) as tmp_file:
        tmp_file.write(binary_bytes)
        tmp_file_path = tmp_file.name
    try:
        result = subprocess.run(
            ["go", "tool", "objdump", tmp_file_path],
            capture_output=True, text=True, check=True, encoding="utf-8"
        )
        return result.stdout
    except subprocess.CalledProcessError:
        return None
    finally:
        os.remove(tmp_file_path)


_RE_JUMP     = re.compile(r'\b(J[A-Z]{1,3})\s+(0x[0-9a-fA-F]+)\b')
_RE_RELOC    = re.compile(r'\[\d+:\d+\](R_[A-Z_]+):(.+)')
_RE_ANGLE    = re.compile(r'<[^>]+>$')
_RE_SIGN_NUM = re.compile(r'[\+\-]\d*$')
_RE_ABS_ADDR = re.compile(r'\b([A-Z]+)\s+0x[0-9a-fA-F]+\b')

_CORE_PREFIXES = frozenset((
    "runtime", "go", "os", "fmt", "syscall", "sync",
    "internal", "reflect", "strings", "strconv", "errors"
))


def clean_instructions(instructions: list[str]) -> str:
    parsed_lines = []
    jump_targets = set()

    for line in instructions:
        parts = line.strip().split(maxsplit=3)
        if len(parts) < 4:
            continue
        line_number = parts[0].split(':')[-1]
        address = parts[1]
        instruction_text = parts[3]
        m = _RE_JUMP.search(instruction_text)
        if m:
            jump_targets.add(m.group(2))
        parsed_lines.append((line_number, address, instruction_text))

    label_map = {addr: f"LABEL_{i+1}" for i, addr in enumerate(sorted(jump_targets))}

    cleaned = []
    for line_number, address, instruction in parsed_lines:
        reloc_match = _RE_RELOC.search(instruction)
        if reloc_match:
            reloc_type = reloc_match.group(1)
            target = _RE_ANGLE.sub('', reloc_match.group(2))
            target = _RE_SIGN_NUM.sub('', target)
            base_inst = instruction[:reloc_match.start()].strip()
            instruction = f"CALL {target}" if reloc_type == 'R_CALL' and base_inst.startswith('CALL ') \
                          else f"{base_inst} {target}"
        else:
            instruction = _RE_SIGN_NUM.sub('', instruction)

        m = _RE_JUMP.search(instruction)
        if m:
            op, target_hex = m.group(1), m.group(2)
            if target_hex in label_map:
                start, end = m.span()
                instruction = instruction[:start] + f"{op} {label_map[target_hex]}" + instruction[end:]
        else:
            instruction = _RE_ABS_ADDR.sub(r'\1', instruction)

        prefix = f"{label_map[address]}: " if address in label_map else ""
        cleaned.append(f"{line_number} {prefix}{instruction.strip()}")

    return "\n".join(cleaned)


def get_assembly_blocks(assembly: str) -> list[AssemblyBlock]:
    blocks = []
    lines = assembly.split('\n')
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if line.startswith('TEXT '):
            directive_part = line.split(' ', 1)[1]
            parts = directive_part.rsplit(' ', 1)
            symbol_name = parts[0]
            filepath = parts[1] if len(parts) > 1 and parts[1].startswith('/') else None
            raw_instructions = []
            i += 1
            while i < len(lines) and not lines[i].strip().startswith('TEXT '):
                if lines[i].strip():
                    raw_instructions.append(lines[i])
                i += 1
            blocks.append(AssemblyBlock(symbol_name, filepath, raw_instructions))
        else:
            i += 1
    return blocks


def get_user_code_from_assembly_code(blocks: list[AssemblyBlock]) -> list[AssemblyBlock]:
    main_block   = next((b for b in blocks if b.symbol_name == 'main.main(SB)'), None)
    plugin_block = next((b for b in blocks if b.symbol_name.startswith('plugin/')), None)

    main_top   = main_block.filepath.split('/')[1]   if (main_block   and main_block.filepath) else None
    plugin_top = plugin_block.filepath.split('/')[1] if (plugin_block and plugin_block.filepath) else None

    filtered = []
    for block in blocks:
        fp = block.filepath
        if fp is None:
            continue
        if block.symbol_name.split(".")[0] in _CORE_PREFIXES:
            continue
        if main_top is not None and fp.split('/')[1] != main_top:
            continue
        if plugin_top is not None and fp.split('/')[1] != plugin_top:
            continue
        filtered.append(block)
    return filtered


def chunked_iterable(iterable, size):
    it = iter(iterable)
    for first in it:
        yield [first] + list(islice(it, size - 1))


## Generate and Process Optimized Assembly

In [ ]:
FINAL_COMPLETIONS_PATH = Path("results") / "final_completions.jsonl"
ORIGINAL_FORMAT_PATH   = Path("datasets") / "objdump_original_format_source_only.jsonl"
LABELED_FORMAT_PATH    = Path("datasets") / "objdump_labeled_format_source_only.jsonl"
ORIGINAL_FOLDER        = Path("datasets") / "objdump_original_format_source_only"
LABELED_FOLDER         = Path("datasets") / "objdump_labeled_format_source_only"

ORIGINAL_FOLDER.mkdir(exist_ok=True)
LABELED_FOLDER.mkdir(exist_ok=True)

# Resume from where a previous run left off; avoids recompiling already-processed entries
processed_count = sum(1 for _ in ORIGINAL_FORMAT_PATH.open(encoding="utf-8")) \
                  if ORIGINAL_FORMAT_PATH.exists() else 0

# Number of rows processed per parallel batch — balances memory use and parallelisation overhead
CHUNK_SIZE = 50


def _process_optimized(raw_row: str) -> tuple | None:
    """Compile one Go snippet and extract user-level assembly.

    Returns (original_row, labeled_row, original_asm, labeled_asm) on success,
    or None if compilation or objdump fails.
    """
    row = json.loads(raw_row)
    binary = get_compiled_code(row["code"], row["category"], "asm")
    if binary is None:
        return None
    assembly = get_assembly_code(binary)
    if assembly is None:
        return None
    blocks = get_assembly_blocks(assembly)
    user_blocks = get_user_code_from_assembly_code(blocks)
    original = "\n\n".join(b.original_format() for b in user_blocks)
    labeled  = "\n\n".join(b.labeled_format()  for b in user_blocks)
    return {**row, "compiled_asm": original}, {**row, "compiled_asm": labeled}, original, labeled


with open(FINAL_COMPLETIONS_PATH, "r", encoding="utf-8") as f_in, \
     open(ORIGINAL_FORMAT_PATH, "a", encoding="utf-8") as f_orig, \
     open(LABELED_FORMAT_PATH,  "a", encoding="utf-8") as f_lab:

    chunks = chunked_iterable(islice(f_in, processed_count, None), CHUNK_SIZE)

    with Parallel(n_jobs=-1, backend="loky") as parallel:
        for chunk in tqdm(chunks, desc="Compiling (optimized)"):
            for result in parallel(delayed(_process_optimized)(line) for line in chunk):
                if result is None:
                    continue
                row_orig, row_lab, original, labeled = result
                f_orig.write(json.dumps(row_orig) + "\n")
                f_lab.write(json.dumps(row_lab) + "\n")
                pid, mod = row_orig["prompt_id"], row_orig["modifier"]
                (ORIGINAL_FOLDER / f"{pid}_{mod}_original.txt").write_text(original, encoding="utf-8")
                (LABELED_FOLDER  / f"{pid}_{mod}_with_labels.txt").write_text(labeled, encoding="utf-8")


## Generate and Process Non-Optimized Assembly

In [ ]:
NOT_OPT_LABELED_PATH   = Path("datasets") / "not_optimized_objdump_labeled_format_source_only.jsonl"
NOT_OPT_LABELED_FOLDER = Path("datasets") / "not_optimized_objdump_labeled_format_source_only"
NOT_OPT_LABELED_FOLDER.mkdir(exist_ok=True)

processed_count = sum(1 for _ in NOT_OPT_LABELED_PATH.open(encoding="utf-8")) \
                  if NOT_OPT_LABELED_PATH.exists() else 0


def _process_not_optimized(raw_row: str) -> tuple | None:
    """Same as _process_optimized but compiles without optimizations or inlining.

    Returns (row, labeled_asm) on success, or None if compilation or objdump fails.
    """
    row = json.loads(raw_row)
    # -N disables optimizations; -l disables inlining — preserves original code structure in assembly
    binary = get_compiled_code(row["code"], row["category"], "asm", gcflags="all=-N -l")
    if binary is None:
        return None
    assembly = get_assembly_code(binary)
    if assembly is None:
        return None
    blocks = get_assembly_blocks(assembly)
    user_blocks = get_user_code_from_assembly_code(blocks)
    labeled = "\n\n".join(b.labeled_format() for b in user_blocks)
    return {**row, "compiled_asm": labeled}, labeled


with open(FINAL_COMPLETIONS_PATH, "r", encoding="utf-8") as f_in, \
     open(NOT_OPT_LABELED_PATH, "a", encoding="utf-8") as f_out:

    chunks = chunked_iterable(islice(f_in, processed_count, None), CHUNK_SIZE)

    with Parallel(n_jobs=-1, backend="loky") as parallel:
        for chunk in tqdm(chunks, desc="Compiling (not optimized)"):
            for result in parallel(delayed(_process_not_optimized)(line) for line in chunk):
                if result is None:
                    continue
                row, labeled = result
                f_out.write(json.dumps(row) + "\n")
                pid, mod = row["prompt_id"], row["modifier"]
                (NOT_OPT_LABELED_FOLDER / f"{pid}_{mod}_with_labels.txt").write_text(labeled, encoding="utf-8")


## Merge All Variants into one file

In [ ]:
import json
import os


labeled_dict = {}
with open("datasets/objdump_labeled_format_source_only.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        key = (data["prompt_id"], data["model"], data["modifier"])
        labeled_dict[key] = data.get("compiled_asm", None)

not_opt_dict = {}
with open("datasets/not_optimized_objdump_labeled_format_source_only.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        key = (data["prompt_id"], data["model"], data["modifier"])
        not_opt_dict[key] = data.get("compiled_asm", None)

output_file = "datasets/merged_assembly_dataset.jsonl"
with open("datasets/objdump_original_format_source_only.jsonl", "r", encoding="utf-8") as infile, \
     open(output_file, "w", encoding="utf-8") as outfile:

    for line in infile:
        data = json.loads(line)
        key = (data["prompt_id"], data["model"], data["modifier"])

        if "compiled_asm" in data:
            # Rename to clarify that assembly is linked back to the original source file structure
            data["source_linked_assembly"] = data.pop("compiled_asm")

        data["simplified_source_linked_assembly"] = labeled_dict.get(key, None)
        data["non_optimized_simplified_source_linked_assembly"] = not_opt_dict.get(key, None)

        outfile.write(json.dumps(data, ensure_ascii=False) + "\n")

print(f"Merged dataset saved to: {output_file}")
